# 🥑 Notebook 02: Approach 1 — Classical Computer Vision + Machine Learning
## Avocado Ripening Stage Classification — Greenpods Labs Assignment 2

**Pipeline:**  
Image → CIELAB + HSV Color Histograms + LBP Texture → StandardScaler → SVM / Random Forest → Evaluation

**Why this approach?**
- Avocado ripening is strongly correlated with skin color (green → dark purple/black)
- CIELAB's `a*` channel explicitly measures the green↔red axis — optimal for this task
- LBP captures texture changes: smooth (underripe) → pebbled/bumpy (ripe)
- SVM & Random Forest are fast, interpretable, and work well with hand-crafted features

---

## 1. Setup & Imports

In [ ]:
import sys
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import (
    load_image_paths_and_labels,
    stratified_split,
    build_feature_matrix,
    extract_features_single,
    CLASS_NAMES,
)
from src.classical_model import (
    train_svm,
    train_random_forest,
    evaluate_classifier,
    plot_feature_importance,
    save_model,
    load_model,
    predict_single_image,
)

plt.rcParams.update({'figure.dpi': 120})

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
MODELS_DIR = PROJECT_ROOT / 'models'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete. All modules imported.')


## 2. Data Loading & Stratified Split

In [ ]:
paths, labels, class_names = load_image_paths_and_labels(str(DATA_DIR))
print(f'Total images: {len(paths)}')
print(f'Classes: {class_names}')

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(paths, labels)

print(f'\nSplit sizes:')
print(f'  Train : {len(X_train)}')
print(f'  Val   : {len(X_val)}')
print(f'  Test  : {len(X_test)}')


## 3. Feature Engineering

**Feature vector composition (per image):**

| Component | Description | Dimensions |
|-----------|-------------|------------|
| L* histogram | Lightness channel (CIELAB) | 16 bins |
| a* histogram | Green↔Red axis (CIELAB) | 16 bins |
| b* histogram | Blue↔Yellow axis (CIELAB) | 16 bins |
| H histogram | Hue channel (HSV) | 16 bins |
| S histogram | Saturation channel (HSV) | 16 bins |
| LBP histogram | Local Binary Pattern texture | 10 bins |
| **Total** | | **90 dimensions** |

In [ ]:
sample_path = X_train[0]
sample_feat = extract_features_single(sample_path)
print(f'Feature vector for sample image:')
print(f'  Path  : {sample_path}')
print(f'  Shape : {sample_feat.shape}')
print(f'  Range : [{sample_feat.min():.4f}, {sample_feat.max():.4f}]')
print(f'  Sum   : {sample_feat.sum():.4f} (L1-normalised → should be ≈1.0)')


In [ ]:
print('Extracting features for TRAIN set...')
X_train_feat, y_train_arr = build_feature_matrix(X_train, y_train)

print('\nExtracting features for VAL set...')
X_val_feat, y_val_arr = build_feature_matrix(X_val, y_val)

print('\nExtracting features for TEST set...')
X_test_feat, y_test_arr = build_feature_matrix(X_test, y_test)

print(f'\nFeature matrix shapes:')
print(f'  X_train : {X_train_feat.shape}')
print(f'  X_val   : {X_val_feat.shape}')
print(f'  X_test  : {X_test_feat.shape}')

np.save(PROCESSED_DIR / 'X_train_feat.npy', X_train_feat)
np.save(PROCESSED_DIR / 'X_val_feat.npy',   X_val_feat)
np.save(PROCESSED_DIR / 'X_test_feat.npy',  X_test_feat)
np.save(PROCESSED_DIR / 'y_train.npy',      y_train_arr)
np.save(PROCESSED_DIR / 'y_val.npy',        y_val_arr)
np.save(PROCESSED_DIR / 'y_test.npy',       y_test_arr)
print('\nFeature matrices saved to data/processed/')


## 4. Feature Visualisation — t-SNE (2D Projection)

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

print('Running t-SNE on training features... (may take ~1 min)')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_feat)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_2d = tsne.fit_transform(X_scaled)

stage_colors = {'underripe': '#4CAF50', 'breaking': '#FFC107', 'ripe': '#FF7043', 'overripe': '#8D6E63'}

fig, ax = plt.subplots(figsize=(9, 6))
for i, cls in enumerate(class_names):
    mask = (y_train_arr == i)
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               label=cls.capitalize(), alpha=0.7, s=30,
               color=stage_colors.get(cls, f'C{i}'), edgecolors='none')

ax.set_title('t-SNE Projection of Classical Features', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE Component 1'); ax.set_ylabel('t-SNE Component 2')
ax.legend(fontsize=10, framealpha=0.8)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'tsne_classical_features.png', dpi=150)
plt.show()
print('t-SNE plot saved.')


## 5. Model Training — Support Vector Machine (SVM)

In [ ]:
svm_model = train_svm(
    X_train_feat, y_train_arr,
    C=10.0, kernel='rbf', gamma='scale',
)

print('\n--- Validation Set ---')
svm_val_results = evaluate_classifier(
    svm_model, X_val_feat, y_val_arr,
    class_names=class_names,
    model_name='SVM (Validation)',
    save_dir=str(PROCESSED_DIR),
)


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print('Running GridSearchCV for SVM hyperparameter tuning...')
param_grid = {
    'svm__C':     [1, 10, 100],
    'svm__gamma': ['scale', 'auto'],
}
svm_pipe = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42))])
grid_search = GridSearchCV(svm_pipe, param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_search.fit(X_train_feat, y_train_arr)

print(f'\nBest params : {grid_search.best_params_}')
print(f'Best CV F1  : {grid_search.best_score_:.4f}')
svm_model = grid_search.best_estimator_


In [ ]:
print('\n--- TEST SET EVALUATION (SVM) ---')
svm_test_results = evaluate_classifier(
    svm_model, X_test_feat, y_test_arr,
    class_names=class_names,
    model_name='SVM (Test)',
    save_dir=str(PROCESSED_DIR),
)

save_model(svm_model, str(MODELS_DIR / 'svm_model.joblib'))


## 6. Model Training — Random Forest

In [ ]:
rf_model = train_random_forest(
    X_train_feat, y_train_arr,
    n_estimators=300, max_depth=None,
)

print('\n--- TEST SET EVALUATION (Random Forest) ---')
rf_test_results = evaluate_classifier(
    rf_model, X_test_feat, y_test_arr,
    class_names=class_names,
    model_name='Random Forest (Test)',
    save_dir=str(PROCESSED_DIR),
)

save_model(rf_model, str(MODELS_DIR / 'rf_model.joblib'))


## 7. Feature Importance (Random Forest)

In [ ]:
feature_names = (
    [f'L*_bin{i}' for i in range(16)] +
    [f'a*_bin{i}' for i in range(16)] + 
    [f'b*_bin{i}' for i in range(16)] +
    [f'H_bin{i}'  for i in range(16)] +
    [f'S_bin{i}'  for i in range(16)] +
    [f'LBP_bin{i}' for i in range(10)]
)

plot_feature_importance(rf_model, feature_names=feature_names, top_n=25)


## 8. Learning Curve Analysis

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    svm_model, X_train_feat, y_train_arr,
    cv=3, scoring='f1_macro', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 8)
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(train_sizes,
                train_scores.mean(1) - train_scores.std(1),
                train_scores.mean(1) + train_scores.std(1), alpha=0.15, color='steelblue')
ax.fill_between(train_sizes,
                val_scores.mean(1) - val_scores.std(1),
                val_scores.mean(1) + val_scores.std(1), alpha=0.15, color='coral')
ax.plot(train_sizes, train_scores.mean(1), 'o-', label='Train F1', color='steelblue')
ax.plot(train_sizes, val_scores.mean(1),   's--', label='Val F1',   color='coral')
ax.set_title('SVM Learning Curve', fontsize=13, fontweight='bold')
ax.set_xlabel('Training Samples'); ax.set_ylabel('Macro F1-Score')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'svm_learning_curve.png', dpi=150)
plt.show()


## 9. Single Image Prediction Demo

In [ ]:
import cv2
import matplotlib.pyplot as plt

import random
demo_idx = random.randint(0, len(X_test) - 1)
demo_path = X_test[demo_idx]
true_label = class_names[y_test[demo_idx]]

svm_pred, svm_conf = predict_single_image(svm_model, demo_path, class_names)
rf_pred,  rf_conf  = predict_single_image(rf_model,  demo_path, class_names)

img = cv2.imread(demo_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(img_rgb)
ax.set_title(
    f'True: {true_label}\n'
    f'SVM: {svm_pred} ({svm_conf:.1%}) | RF: {rf_pred} ({rf_conf:.1%})',
    fontsize=11
)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Image: {demo_path}')
print(f'True Label : {true_label}')
print(f'SVM Pred   : {svm_pred} (confidence: {svm_conf:.1%})')
print(f'RF  Pred   : {rf_pred} (confidence: {rf_conf:.1%})')


## 10. Approach 1 Summary

| Model | Test Accuracy | Macro F1 | Inference Speed |
|-------|--------------|----------|-----------------|
| SVM (RBF)     | *see above* | *see above* | ~5 ms/image |
| Random Forest | *see above* | *see above* | ~3 ms/image |

**Key Takeaways:**
- `a*` channel bins are among the most important features (green→red transition)
- SVM with RBF kernel outperforms RF on this low-dimensional feature space
- Class imbalance handled via `class_weight='balanced'`

**➡️ Next: `03_approach_2_deep_learning.ipynb`**